# Acoustic PdM — Phase 4: Notebook 05
## Multi-Model Comparative Evaluation & SOTA Benchmarking

**Project:** Acoustic Predictive Maintenance (Acoustic PdM)  
**Dataset:** Hitachi MIMII — 4 Industrial Machine Types (**Fan, Pump, Slider, Valve**) at 6 dB SNR, ID 00  
**Inputs:** Preprocessed Tensors (from **NB03**) + Model Checkpoints (from **NB04**)  
**Runtime:** GPU Recommended (Kaggle T4 or P100) — Total runtime: ~1–2 minutes  

---

### Key Highlights of this Notebook:

1. **Clip-Level SOTA Aggregation (DCASE / MIMII Standard):**  
   In industrial acoustic monitoring, audio recordings are evaluated at the **10-second WAV file level** by averaging per-block reconstruction errors across all ~309 sliding context frames (160ms each):
   $$\text{Clip Anomaly Score} = \frac{1}{B} \sum_{b=1}^{B} S(\text{block}_b)$$

2. **7-Model Head-to-Head Benchmark on Fan:**
   - Shallow Baselines: **Isolation Forest**, **One-Class SVM**, **XGBoost (Supervised)**
   - Deep Autoencoders: **LSTM-AE**, **FC-AE**, **Conv2D-AE**
   - SOTA Hybrid: **Dual-Stage Deep Hybrid (Conv2D-AE + Latent IF)**

3. **Threshold Calibration ($\theta = P_{95}$):**  
   Calibrated on healthy validation clips to guarantee $\le 5\%$ False Alarm Rate.

4. **Multi-Machine Generalization:**  
   Benchmarking Conv2D-AE and Dual-Stage Hybrid across Fan, Pump, Slider, and Valve.

5. **Explainable AI (XAI):**  
   Spectrogram difference heatmaps $|X - \hat{X}|$ locating physical defect frequencies.

---
### Step 0: Environment Bootstrap & Input Discovery

Auto-detects Kaggle vs Local environments and discovers NB03 processed tensors and NB04 checkpoints.

In [ ]:
import os
import gc
import random
import time
from pathlib import Path
import numpy as np
import torch

# -- Reproducibility Seeds --
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# -- Path Discovery --
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working")
    
    # 1. Discover NB03 processed tensors
    _data_dir = None
    for _root, _dirs, _ in os.walk("/kaggle/input"):
        if "processed" in _dirs:
            _data_dir = Path(_root) / "processed"
            break
        elif "fan" in _dirs and "pump" in _dirs:
            _data_dir = Path(_root)
            break
    PROCESSED_DIR = _data_dir if _data_dir is not None else Path("/kaggle/working/data/processed")

    # 2. Discover NB04 model checkpoints
    _models_dir = None
    for _root, _dirs, _files in os.walk("/kaggle/input"):
        if any(f.endswith(".pth") for f in _files):
            _models_dir = Path(_root)
            break
    if _models_dir is None:
        _models_dir = Path("/kaggle/working/models")
    MODELS_DIR = _models_dir
else:
    _cwd = Path(os.getcwd()).resolve()
    PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
    MODELS_DIR = PROJECT_ROOT / "models"

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("ACOUSTIC PDM -- PHASE 4 EVALUATION ENGINE")
print(f"Environment:    {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Device:         {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Processed Data: {PROCESSED_DIR}")
print(f"Checkpoints:    {MODELS_DIR}")
print(f"Reports Dir:    {REPORTS_DIR}")
print("=" * 70)

---
### Step 1: Import Libraries

Import standard scientific, ML, and visualization packages.

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support

print("Libraries successfully imported.")

---
### Step 2: Neural Architectures & Dataset (Inline Definitions)

Exact architecture definitions matching NB04 for loading `.pth` state dictionaries.

In [ ]:
class AcousticTensorDataset(Dataset):
    def __init__(self, data_array):
        if isinstance(data_array, (str, Path)):
            self.data = np.load(str(data_array)).astype(np.float32)
        else:
            self.data = data_array.astype(np.float32)
        if self.data.ndim == 3:
            self.data = np.expand_dims(self.data, axis=1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx])


class Conv2DAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
        )
        self.encoder_fc = nn.Linear(128 * 16 * 1, latent_dim)
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128 * 16 * 1), nn.LeakyReLU(0.2, inplace=True)
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=(1, 1)),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=(1, 0)),
        )

    def encode(self, x):
        h = self.encoder_conv(x)
        return self.encoder_fc(torch.flatten(h, start_dim=1))

    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 128, 16, 1)
        return self.decoder_conv(h)

    def forward(self, x):
        return self.decode(self.encode(x))


class FCAutoencoder(nn.Module):
    def __init__(self, input_dim=640, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Linear(512, input_dim)
        )

    def encode(self, x):
        return self.encoder(x.view(x.size(0), -1))

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 128, 5)

    def forward(self, x):
        return self.decode(self.encode(x))


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_mels=128, context_frames=5, hidden_dim=64, latent_dim=32):
        super().__init__()
        self.context_frames = context_frames
        self.encoder_lstm = nn.LSTM(input_size=n_mels, hidden_size=hidden_dim, num_layers=2, batch_first=True)
        self.encoder_fc = nn.Linear(hidden_dim, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=2, batch_first=True)
        self.output_fc = nn.Linear(hidden_dim, n_mels)

    def encode(self, x):
        seq = x.squeeze(1).permute(0, 2, 1)
        _, (h_n, _) = self.encoder_lstm(seq)
        return self.encoder_fc(h_n[-1])

    def decode(self, z):
        h = self.decoder_fc(z).unsqueeze(1).repeat(1, self.context_frames, 1)
        out, _ = self.decoder_lstm(h)
        return self.output_fc(out).permute(0, 2, 1).unsqueeze(1)

    def forward(self, x):
        return self.decode(self.encode(x))


print(f"Architectures initialized (Conv2D-AE: {sum(p.numel() for p in Conv2DAutoencoder().parameters()):,} params)")

---
### Step 3: Load Data Partitions & Trained Model Weights

Loads preprocessed tensors and `.pth` / `.joblib` checkpoints for all models.

In [ ]:
MACHINES = ["fan", "pump", "slider", "valve"]

def load_machine_tensors(machine_name, base_dir=PROCESSED_DIR):
    m_dir = Path(base_dir) / machine_name
    train_normal = np.load(m_dir / "train_normal.npy").astype(np.float32)
    val_normal   = np.load(m_dir / "val_normal.npy").astype(np.float32)
    test_normal  = np.load(m_dir / "test_normal.npy").astype(np.float32)
    test_anomaly = np.load(m_dir / "test_anomaly.npy").astype(np.float32)
    return train_normal, val_normal, test_normal, test_anomaly

# -- Load Conv2D-AE models (4 machines) --
conv2d_models = {}
for machine in MACHINES:
    m = Conv2DAutoencoder(latent_dim=32)
    m.load_state_dict(torch.load(str(MODELS_DIR / f"best_conv2d_ae_{machine}.pth"), map_location=DEVICE))
    m = m.to(DEVICE).eval()
    conv2d_models[machine] = m
    print(f"  Conv2D-AE [{machine.upper()}] loaded")

# -- Load Fan baseline models --
fc_model = FCAutoencoder(input_dim=640, latent_dim=32)
fc_model.load_state_dict(torch.load(str(MODELS_DIR / "best_fc_ae_fan.pth"), map_location=DEVICE))
fc_model = fc_model.to(DEVICE).eval()
print("  FC-AE [FAN] loaded")

lstm_model = LSTMAutoencoder(n_mels=128, context_frames=5, hidden_dim=64, latent_dim=32)
lstm_model.load_state_dict(torch.load(str(MODELS_DIR / "best_lstm_ae_fan.pth"), map_location=DEVICE))
lstm_model = lstm_model.to(DEVICE).eval()
print("  LSTM-AE [FAN] loaded")

# -- Load Shallow baselines --
if_shallow = joblib.load(str(MODELS_DIR / "shallow_iforest_fan.joblib"))
ocsvm      = joblib.load(str(MODELS_DIR / "shallow_ocsvm_fan.joblib"))
xgb_clf    = joblib.load(str(MODELS_DIR / "supervised_xgb_fan.joblib"))
print("  Shallow baselines (IF, OC-SVM, XGBoost) loaded")

# -- Load Hybrid Latent IF models (4 machines) --
hybrid_if_models = {}
for machine in MACHINES:
    hybrid_if_models[machine] = joblib.load(str(MODELS_DIR / f"hybrid_latent_if_{machine}.joblib"))
print("  Hybrid Latent IF models loaded (4 machines)")

print("\nAll 11 model checkpoints loaded successfully.")

---
### Step 4: Scoring & Clip-Level Aggregation Utilities (SOTA Protocol)

Implements the official **DCASE / MIMII evaluation protocol**:
- **Reconstruction MSE:** $S_{\text{recon}}$
- **Latent Manifold Outlier Score:** $S_{\text{latent}} = -\text{score\_samples}(z)$
- **Dual-Stage Fusion:** $S_{\text{hybrid}} = 0.6 \cdot \tilde{S}_{\text{recon}} + 0.4 \cdot \tilde{S}_{\text{latent}}$
- **Clip-Level Aggregation:** $S_{\text{clip}} = \frac{1}{B} \sum_{b=1}^{B} S(\text{block}_b)$ across all ~309 blocks of each 10-second file
- **Threshold Calibration:** $\theta = P_{95}$ on normal validation clips

In [ ]:
def compute_reconstruction_error(model, data_array, batch_size=256, device=DEVICE):
    """Compute per-block MSE reconstruction error."""
    model = model.to(device).eval()
    loader = DataLoader(AcousticTensorDataset(data_array), batch_size=batch_size, shuffle=False)
    errors = []
    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            recon = model(x)
            mse = ((x - recon) ** 2).view(x.size(0), -1).mean(dim=1)
            errors.append(mse.cpu().numpy())
    return np.concatenate(errors, axis=0)

def extract_latents(model, data_array, batch_size=256, device=DEVICE):
    """Extract 32-dim latent representations z from autoencoder bottleneck."""
    model = model.to(device).eval()
    loader = DataLoader(AcousticTensorDataset(data_array), batch_size=batch_size, shuffle=False)
    latents = []
    with torch.no_grad():
        for x in loader:
            z = model.encode(x.to(device))
            latents.append(z.cpu().numpy())
    return np.concatenate(latents, axis=0)

def compute_hybrid_score(s_recon, s_latent, alpha=0.6):
    """Fuse Min-Max normalized physical and latent outlier scores."""
    def min_max(arr):
        denom = arr.max() - arr.min()
        return (arr - arr.min()) / (denom if denom > 1e-10 else 1.0)
    return alpha * min_max(s_recon) + (1.0 - alpha) * min_max(s_latent)

def aggregate_blocks_to_clips(block_scores, blocks_per_clip=309):
    """
    Aggregate per-block scores to per-clip (10s WAV) anomaly scores.
    Official DCASE / MIMII SOTA evaluation protocol.
    """
    n_total = len(block_scores)
    n_clips = max(1, round(n_total / blocks_per_clip))
    clip_chunks = np.array_split(block_scores, n_clips)
    
    clip_scores = [float(np.mean(chunk)) for chunk in clip_chunks if len(chunk) > 0]
    return np.array(clip_scores, dtype=np.float32)

def calibrate_threshold(val_normal_scores, percentile=95.0):
    """Calibrate theta = P_95 on healthy validation clips (guarantees <= 5% FPR)."""
    return float(np.percentile(val_normal_scores, percentile))

def compute_metrics(y_true, y_scores, threshold=None):
    """Compute ROC-AUC, pAUC (10% max FPR), and Precision/Recall/F1."""
    auc = roc_auc_score(y_true, y_scores)
    pauc = roc_auc_score(y_true, y_scores, max_fpr=0.1)
    res = {"ROC-AUC": round(float(auc), 4), "pAUC (10%)": round(float(pauc), 4)}
    if threshold is not None:
        y_pred = (y_scores >= threshold).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        res.update({"Precision": round(float(p), 4), "Recall": round(float(r), 4), "F1": round(float(f1), 4)})
    return res

print("Scoring engines and Clip-Level aggregation functions defined.")

---
### Step 5: Compute SOTA Anomaly Scores — Fan 7-Model Benchmark

Scoring all 7 models at the clip level on the **Fan** evaluation dataset.

In [ ]:
print("=" * 70)
print("  COMPUTING CLIP-LEVEL ANOMALY SCORES -- FAN BENCHMARK (7 MODELS)")
print("=" * 70)

_, val_fan, test_fan_norm, test_fan_anom = load_machine_tensors("fan")
BLOCKS_PER_CLIP = 309

val_flat       = val_fan.reshape(len(val_fan), -1)
test_norm_flat = test_fan_norm.reshape(len(test_fan_norm), -1)
test_anom_flat = test_fan_anom.reshape(len(test_fan_anom), -1)

n_v_clips = max(1, round(len(val_fan) / BLOCKS_PER_CLIP))
n_n_clips = max(1, round(len(test_fan_norm) / BLOCKS_PER_CLIP))
n_a_clips = max(1, round(len(test_fan_anom) / BLOCKS_PER_CLIP))
y_true_clips = np.array([0] * n_n_clips + [1] * n_a_clips)

print("Fan Dataset Partitions:")
print(f"  Normal Validation: {len(val_fan):,} blocks (~{n_v_clips} clips)")
print(f"  Test Normal:       {len(test_fan_norm):,} blocks (~{n_n_clips} clips)")
print(f"  Test Anomaly:      {len(test_fan_anom):,} blocks (~{n_a_clips} clips)")
print(f"  Total Test Files:  {n_n_clips + n_a_clips} audio clips")

fan_scores = {}

# 1. Isolation Forest
print("\n1. Isolation Forest (Shallow)...")
t0 = time.time()
s_if_norm = aggregate_blocks_to_clips(-if_shallow.score_samples(test_norm_flat), BLOCKS_PER_CLIP)
s_if_anom = aggregate_blocks_to_clips(-if_shallow.score_samples(test_anom_flat), BLOCKS_PER_CLIP)
s_if_val  = aggregate_blocks_to_clips(-if_shallow.score_samples(val_flat), BLOCKS_PER_CLIP)
fan_scores["Isolation Forest"] = {"clip_scores": np.concatenate([s_if_norm, s_if_anom]), "val_clips": s_if_val}
print(f"   Done in {time.time()-t0:.1f}s")

# 2. One-Class SVM
print("2. One-Class SVM (Shallow)...")
t0 = time.time()
s_svm_norm = aggregate_blocks_to_clips(-ocsvm.decision_function(test_norm_flat), BLOCKS_PER_CLIP)
s_svm_anom = aggregate_blocks_to_clips(-ocsvm.decision_function(test_anom_flat), BLOCKS_PER_CLIP)
s_svm_val  = aggregate_blocks_to_clips(-ocsvm.decision_function(val_flat), BLOCKS_PER_CLIP)
fan_scores["One-Class SVM"] = {"clip_scores": np.concatenate([s_svm_norm, s_svm_anom]), "val_clips": s_svm_val}
print(f"   Done in {time.time()-t0:.1f}s")

# 3. XGBoost
print("3. XGBoost (Supervised)...")
t0 = time.time()
s_xgb_norm = aggregate_blocks_to_clips(xgb_clf.predict_proba(test_norm_flat)[:, 1], BLOCKS_PER_CLIP)
s_xgb_anom = aggregate_blocks_to_clips(xgb_clf.predict_proba(test_anom_flat)[:, 1], BLOCKS_PER_CLIP)
s_xgb_val  = aggregate_blocks_to_clips(xgb_clf.predict_proba(val_flat)[:, 1], BLOCKS_PER_CLIP)
fan_scores["XGBoost (Supervised)"] = {"clip_scores": np.concatenate([s_xgb_norm, s_xgb_anom]), "val_clips": s_xgb_val}
print(f"   Done in {time.time()-t0:.1f}s")

# 4. LSTM-AE
print("4. LSTM-AE...")
t0 = time.time()
s_lstm_norm = aggregate_blocks_to_clips(compute_reconstruction_error(lstm_model, test_fan_norm), BLOCKS_PER_CLIP)
s_lstm_anom = aggregate_blocks_to_clips(compute_reconstruction_error(lstm_model, test_fan_anom), BLOCKS_PER_CLIP)
s_lstm_val  = aggregate_blocks_to_clips(compute_reconstruction_error(lstm_model, val_fan), BLOCKS_PER_CLIP)
fan_scores["LSTM-AE"] = {"clip_scores": np.concatenate([s_lstm_norm, s_lstm_anom]), "val_clips": s_lstm_val}
print(f"   Done in {time.time()-t0:.1f}s")

# 5. FC-AE
print("5. FC-AE...")
t0 = time.time()
s_fc_norm = aggregate_blocks_to_clips(compute_reconstruction_error(fc_model, test_fan_norm), BLOCKS_PER_CLIP)
s_fc_anom = aggregate_blocks_to_clips(compute_reconstruction_error(fc_model, test_fan_anom), BLOCKS_PER_CLIP)
s_fc_val  = aggregate_blocks_to_clips(compute_reconstruction_error(fc_model, val_fan), BLOCKS_PER_CLIP)
fan_scores["FC-AE"] = {"clip_scores": np.concatenate([s_fc_norm, s_fc_anom]), "val_clips": s_fc_val}
print(f"   Done in {time.time()-t0:.1f}s")

# 6. Conv2D-AE (Standalone)
print("6. Conv2D-AE (Standalone)...")
t0 = time.time()
s_c2d_norm_b = compute_reconstruction_error(conv2d_models["fan"], test_fan_norm)
s_c2d_anom_b = compute_reconstruction_error(conv2d_models["fan"], test_fan_anom)
s_c2d_val_b  = compute_reconstruction_error(conv2d_models["fan"], val_fan)

s_c2d_norm = aggregate_blocks_to_clips(s_c2d_norm_b, BLOCKS_PER_CLIP)
s_c2d_anom = aggregate_blocks_to_clips(s_c2d_anom_b, BLOCKS_PER_CLIP)
s_c2d_val  = aggregate_blocks_to_clips(s_c2d_val_b, BLOCKS_PER_CLIP)
fan_scores["Conv2D-AE"] = {"clip_scores": np.concatenate([s_c2d_norm, s_c2d_anom]), "val_clips": s_c2d_val}
print(f"   Done in {time.time()-t0:.1f}s")

# 7. Dual-Stage Hybrid (Conv2D-AE + Latent IF)
print("7. Dual-Stage Hybrid (Conv2D-AE + Latent IF)...")
t0 = time.time()
z_norm = extract_latents(conv2d_models["fan"], test_fan_norm)
z_anom = extract_latents(conv2d_models["fan"], test_fan_anom)
z_val  = extract_latents(conv2d_models["fan"], val_fan)

s_lat_norm_b = -hybrid_if_models["fan"].score_samples(z_norm)
s_lat_anom_b = -hybrid_if_models["fan"].score_samples(z_anom)
s_lat_val_b  = -hybrid_if_models["fan"].score_samples(z_val)

s_hyb_norm_b = compute_hybrid_score(s_c2d_norm_b, s_lat_norm_b, alpha=0.6)
s_hyb_anom_b = compute_hybrid_score(s_c2d_anom_b, s_lat_anom_b, alpha=0.6)
s_hyb_val_b  = compute_hybrid_score(s_c2d_val_b, s_lat_val_b, alpha=0.6)

s_hyb_norm = aggregate_blocks_to_clips(s_hyb_norm_b, BLOCKS_PER_CLIP)
s_hyb_anom = aggregate_blocks_to_clips(s_hyb_anom_b, BLOCKS_PER_CLIP)
s_hyb_val  = aggregate_blocks_to_clips(s_hyb_val_b, BLOCKS_PER_CLIP)

fan_scores["Hybrid (Conv2D-AE + IF)"] = {"clip_scores": np.concatenate([s_hyb_norm, s_hyb_anom]), "val_clips": s_hyb_val}
print(f"   Done in {time.time()-t0:.1f}s")

print("\nAll 7 models scored successfully at Clip Level.")

---
### Step 6: Cross-Model Comparison Table

Formats metrics across all 7 models at calibrated $\theta = P_{95}$ and saves `reports/model_comparison_table.csv`.

In [ ]:
MODEL_ORDER = [
    "Isolation Forest",
    "One-Class SVM",
    "XGBoost (Supervised)",
    "LSTM-AE",
    "FC-AE",
    "Conv2D-AE",
    "Hybrid (Conv2D-AE + IF)",
]

results = []

for name in MODEL_ORDER:
    data = fan_scores[name]
    theta = calibrate_threshold(data["val_clips"], percentile=95.0)
    m = compute_metrics(y_true_clips, data["clip_scores"], threshold=theta)
    m["Model"] = name
    m["Threshold"] = round(theta, 6)
    results.append(m)

df_results = pd.DataFrame(results)[["Model", "ROC-AUC", "pAUC (10%)", "Precision", "Recall", "F1", "Threshold"]]

csv_path = REPORTS_DIR / "model_comparison_table.csv"
df_results.to_csv(csv_path, index=False)

print("=" * 90)
print("  CROSS-MODEL BENCHMARK COMPARISON TABLE -- FAN (6 dB SNR, CLIP-LEVEL SOTA)")
print("=" * 90)
print(df_results.to_string(index=False))
print("=" * 90)
print(f"Saved to: {csv_path.name}")

---
### Step 7: Multi-Model ROC Curve Comparison

Generates publication-quality ROC curves comparing all 7 models and saves `reports/multi_model_roc_curves.png`.

In [ ]:
MODEL_COLORS = {
    "Isolation Forest": "#d62728",
    "One-Class SVM": "#9467bd",
    "XGBoost (Supervised)": "#8c564b",
    "LSTM-AE": "#2ca02c",
    "FC-AE": "#ff7f0e",
    "Conv2D-AE": "#1f77b4",
    "Hybrid (Conv2D-AE + IF)": "#e7298a",
}

MODEL_STYLES = {
    "Isolation Forest": {"lw": 1.5, "ls": ":"},
    "One-Class SVM": {"lw": 1.5, "ls": ":"},
    "XGBoost (Supervised)": {"lw": 1.5, "ls": "-."},
    "LSTM-AE": {"lw": 2, "ls": "--"},
    "FC-AE": {"lw": 2, "ls": "--"},
    "Conv2D-AE": {"lw": 2.5, "ls": "-"},
    "Hybrid (Conv2D-AE + IF)": {"lw": 3, "ls": "-"},
}

fig, ax = plt.subplots(figsize=(10, 8))

for name in MODEL_ORDER:
    scores = fan_scores[name]["clip_scores"]
    fpr, tpr, _ = roc_curve(y_true_clips, scores)
    auc_val = roc_auc_score(y_true_clips, scores)
    st = MODEL_STYLES[name]
    ax.plot(fpr, tpr, color=MODEL_COLORS[name], lw=st["lw"], linestyle=st["ls"],
            label=f"{name} (AUC={auc_val:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label="Random Chance (AUC=0.500)")
ax.set_xlabel("False Positive Rate (FPR)", fontsize=13)
ax.set_ylabel("True Positive Rate (TPR)", fontsize=13)
ax.set_title("Multi-Model ROC Curve Comparison -- Fan (6 dB SNR, Clip-Level)", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
roc_path = REPORTS_DIR / "multi_model_roc_curves.png"
plt.savefig(roc_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved ROC curves to: {roc_path.name}")

---
### Step 8: Multi-Machine Generalization (Conv2D-AE + Dual-Stage Hybrid)

Evaluates Conv2D-AE and Hybrid across all 4 machine types (**Fan, Pump, Slider, Valve**) and saves `reports/multi_machine_hybrid_results.csv`.

In [ ]:
print("=" * 70)
print("  MULTI-MACHINE EVALUATION (Conv2D-AE + HYBRID -- CLIP LEVEL)")
print("=" * 70)

multi_results = []

for machine in MACHINES:
    print(f"\nEvaluating: {machine.upper()}...")
    _, val_m, test_m_norm, test_m_anom = load_machine_tensors(machine)

    n_v = max(1, round(len(val_m) / BLOCKS_PER_CLIP))
    n_n = max(1, round(len(test_m_norm) / BLOCKS_PER_CLIP))
    n_a = max(1, round(len(test_m_anom) / BLOCKS_PER_CLIP))
    y_m = np.array([0] * n_n + [1] * n_a)

    # 1. Conv2D-AE standalone
    s_r_n_b = compute_reconstruction_error(conv2d_models[machine], test_m_norm)
    s_r_a_b = compute_reconstruction_error(conv2d_models[machine], test_m_anom)
    s_r_v_b = compute_reconstruction_error(conv2d_models[machine], val_m)

    s_r_n = aggregate_blocks_to_clips(s_r_n_b, BLOCKS_PER_CLIP)
    s_r_a = aggregate_blocks_to_clips(s_r_a_b, BLOCKS_PER_CLIP)
    s_r_v = aggregate_blocks_to_clips(s_r_v_b, BLOCKS_PER_CLIP)

    theta_conv = calibrate_threshold(s_r_v)
    m_conv = compute_metrics(y_m, np.concatenate([s_r_n, s_r_a]), theta_conv)

    # 2. Dual-Stage Hybrid
    z_n = extract_latents(conv2d_models[machine], test_m_norm)
    z_a = extract_latents(conv2d_models[machine], test_m_anom)
    z_v = extract_latents(conv2d_models[machine], val_m)

    sl_n_b = -hybrid_if_models[machine].score_samples(z_n)
    sl_a_b = -hybrid_if_models[machine].score_samples(z_a)
    sl_v_b = -hybrid_if_models[machine].score_samples(z_v)

    s_hyb_n_b = compute_hybrid_score(s_r_n_b, sl_n_b, alpha=0.6)
    s_hyb_a_b = compute_hybrid_score(s_r_a_b, sl_a_b, alpha=0.6)
    s_hyb_v_b = compute_hybrid_score(s_r_v_b, sl_v_b, alpha=0.6)

    s_hyb_n = aggregate_blocks_to_clips(s_hyb_n_b, BLOCKS_PER_CLIP)
    s_hyb_a = aggregate_blocks_to_clips(s_hyb_a_b, BLOCKS_PER_CLIP)
    s_hyb_v = aggregate_blocks_to_clips(s_hyb_v_b, BLOCKS_PER_CLIP)

    theta_hyb = calibrate_threshold(s_hyb_v)
    m_hyb = compute_metrics(y_m, np.concatenate([s_hyb_n, s_hyb_a]), theta_hyb)

    multi_results.append({
        "Machine": machine.upper(),
        "Conv2D-AE AUC": m_conv["ROC-AUC"],
        "Conv2D-AE F1": m_conv["F1"],
        "Hybrid AUC": m_hyb["ROC-AUC"],
        "Hybrid pAUC": m_hyb["pAUC (10%)"],
        "Hybrid F1": m_hyb["F1"],
    })
    print(f"  Conv2D-AE AUC: {m_conv['ROC-AUC']:.4f} | Hybrid AUC: {m_hyb['ROC-AUC']:.4f} | Hybrid F1: {m_hyb['F1']:.4f}")

    del val_m, test_m_norm, test_m_anom, z_n, z_a, z_v
    gc.collect()

df_multi = pd.DataFrame(multi_results)
multi_csv = REPORTS_DIR / "multi_machine_hybrid_results.csv"
df_multi.to_csv(multi_csv, index=False)

print("\n" + "=" * 70)
print(df_multi.to_string(index=False))
print("=" * 70)
print(f"Saved to: {multi_csv.name}")

---
### Step 9: Explainable AI (XAI) — Difference Spectrogram Heatmaps

Generates $D = |X - \hat{X}|$ heatmaps showing localized physical anomaly frequencies across all 4 machines and saves `reports/xai_difference_heatmaps.png`.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 20))

for row, machine in enumerate(MACHINES):
    _, _, _, test_anom = load_machine_tensors(machine)
    ae = conv2d_models[machine]

    sample_idx = len(test_anom) // 3
    sample = test_anom[sample_idx:sample_idx+1]
    sample_t = torch.from_numpy(sample).to(DEVICE)

    ae.eval()
    with torch.no_grad():
        recon_t = ae(sample_t)
    recon = recon_t.cpu().numpy()

    orig = sample.squeeze()
    reconstructed = recon.squeeze()
    diff = np.abs(orig - reconstructed)

    # Column 1: Original
    ax0 = axes[row, 0]
    im0 = ax0.imshow(orig, aspect="auto", origin="lower", cmap="magma")
    ax0.set_title(f"{machine.upper()} -- Original Spectrogram (Anomaly)", fontsize=11, fontweight="bold")
    ax0.set_ylabel("Mel Frequency Bin")
    plt.colorbar(im0, ax=ax0, fraction=0.046)

    # Column 2: Reconstructed
    ax1 = axes[row, 1]
    im1 = ax1.imshow(reconstructed, aspect="auto", origin="lower", cmap="magma")
    ax1.set_title(f"{machine.upper()} -- Reconstructed Spectrogram", fontsize=11, fontweight="bold")
    plt.colorbar(im1, ax=ax1, fraction=0.046)

    # Column 3: XAI Difference Heatmap
    ax2 = axes[row, 2]
    im2 = ax2.imshow(diff, aspect="auto", origin="lower", cmap="hot")
    ax2.set_title(f"{machine.upper()} -- |X - X^| (XAI Defect Heatmap)", fontsize=11, fontweight="bold")
    plt.colorbar(im2, ax=ax2, fraction=0.046)

    if row == 3:
        for ax in axes[row]:
            ax.set_xlabel("Context Time Frame")

    del test_anom
    gc.collect()

plt.tight_layout()
xai_path = REPORTS_DIR / "xai_difference_heatmaps.png"
plt.savefig(xai_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved XAI heatmaps to: {xai_path.name}")

---
### Step 10: Phase 4 Summary & Deliverables Audit

Final audit verifying all generated benchmarks and reports.

In [ ]:
print("=" * 75)
print("PHASE 4 EVALUATION SUMMARY")
print("=" * 75)

best_row = df_results.loc[df_results["ROC-AUC"].idxmax()]
print(f"\nBest Model (Fan): {best_row['Model']}")
print(f"   ROC-AUC:   {best_row['ROC-AUC']:.4f}")
print(f"   pAUC(10%): {best_row['pAUC (10%)']:.4f}")
print(f"   F1-Score:  {best_row['F1']:.4f}")

print(f"\nGenerated Reports in '{REPORTS_DIR}':")
for f in sorted(REPORTS_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name:45s} ({size_kb:.1f} KB)")

print("\n" + "=" * 75)
print("PHASE 4 COMPLETE: All models benchmarked. Ready for NB06 Interactive Demo!")
print("=" * 75)